# Baseline pha 1 — chạy trên Kaggle CPU

Notebook này đo API bằng k6 trong **kernel Kaggle thực**, không có kết quả mẫu được điền sẵn.

1. Trong Kaggle Notebook chọn **Accelerator: None** và **Internet: On**.
2. Thêm Input gồm `source.tar.gz` và `source.tar.gz.sha256` do `node benchmarks/package.cjs` tạo. Archive chỉ chứa code, lockfile và manifest; không chứa `.env`, database, tài khoản thật hoặc Git credentials.
3. Điền đường dẫn ở cell tham số nếu có nhiều archive. Chạy **Run All**.
4. Giữ summary JSON, report Markdown và `runtime.json` trong Output để đối chiếu pha 2.

Runtime cố định: Node **24.12.0**, k6 **1.6.1**, MongoDB **8.0.30** Ubuntu2204/x86_64. Bootstrap đối chiếu SHA256 với nguồn chính thức, ghi phiên bản thực tế và kiểm tra không có GPU/TPU. Mongo chạy riêng ở loopback `127.0.0.1:27018`, replica set `rs0`; mỗi lần có database benchmark mới. Không cần Docker hoặc dữ liệu ứng dụng thật.

Tải runtime và `npm ci` nằm ngoài cửa sổ đo. Runner hiện đo 10 người dùng đồng thời, warmup 10 giây và đo 60 giây; bốn GET có JWT. Số liệu của lần đo thực mới được xuất ra.

Nguồn runtime: [Node checksum](https://nodejs.org/dist/v24.12.0/SHASUMS256.txt), [k6 checksum](https://github.com/grafana/k6/releases/download/v1.6.1/k6-v1.6.1-checksums.txt), [Mongo checksum](https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-8.0.30.tgz.sha256).


In [ ]:
from pathlib import Path

# Leave None only when exactly one source.tar.gz is attached in Kaggle Inputs.
SOURCE_ARCHIVE = None
# Paste the package SHA256 here, or attach its trusted source.tar.gz.sha256 sidecar.
REQUIRED_SHA256 = None


In [ ]:
import hashlib
import os
import re
import sys
import tarfile

working = Path("/kaggle/working")
if sys.platform != "linux" or not working.is_dir() or not any(key.startswith("KAGGLE_") for key in os.environ):
    raise RuntimeError("Run this notebook on Kaggle; local measurements cannot be labeled Kaggle.")
if SOURCE_ARCHIVE is None:
    candidates = list(Path("/kaggle/input").rglob("source.tar.gz")) + list(Path("/kaggle/input").rglob("source.tar.gz.bundle"))
    if len(candidates) != 1:
        raise RuntimeError("Set SOURCE_ARCHIVE to the exact attached source.tar.gz path.")
    archive = candidates[0]
else:
    archive = Path(SOURCE_ARCHIVE)
archive = archive.resolve()
if not archive.is_file():
    raise FileNotFoundError("The attached source archive was not found.")
expected = REQUIRED_SHA256
if not expected:
    checksum = archive.with_name(archive.name + ".sha256")
    if not checksum.is_file():
        raise RuntimeError("Attach the trusted .sha256 sidecar or fill REQUIRED_SHA256.")
    parts = checksum.read_text(encoding="utf-8").strip().split()
    expected = parts[0] if parts else ""
if not re.fullmatch(r"[a-fA-F0-9]{64}", str(expected)):
    raise ValueError("A complete 64-character archive SHA256 is required.")
digest = hashlib.sha256()
with archive.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != expected.lower():
    raise ValueError("Source archive SHA256 mismatch; no archive code was executed.")

# Read only the bootstrap from the checked archive; it validates/extracts the sanitized source.
with tarfile.open(archive, "r:gz") as bundle:
    member = bundle.getmember("benchmarks/kaggle-bootstrap.py")
    if not member.isfile():
        raise ValueError("Bootstrap must be a regular file in the verified source archive.")
    payload = bundle.extractfile(member).read()
tools = working / ".online-exam-baseline-tools"
tools.mkdir(exist_ok=True)
bootstrap = tools / ("kaggle-bootstrap-" + expected.lower()[:12] + ".py")
bootstrap.write_bytes(payload)
print("Verified source archive SHA256:", expected.lower())


In [ ]:
import subprocess

# Installs pinned runtimes, npm ci, owns loopback Mongo27018/rs0, then calls the real runner.
# The runner performs Prisma generate/db push/build, seeds its isolated benchmark DB,
# runs 10 VUs (10s warmup + 60s measured), validates the report and cleans its owned IDs.
subprocess.run([
    sys.executable, str(bootstrap),
    "--source-archive", str(archive),
    "--archive-sha256", expected.lower(),
], check=True)


In [ ]:
from pathlib import Path
import json
import base64
import html
from IPython.display import HTML, Markdown, display

working = Path("/kaggle/working")
root = working / "phase1-baseline-results"
completed = [
    folder for folder in root.iterdir()
    if folder.is_dir() and (folder / "runtime.json").is_file()
    and json.loads((folder / "runtime.json").read_text())["status"] == "completed"
]
if not completed:
    raise RuntimeError("There is no completed, measured Kaggle CPU report.")
latest = max(completed, key=lambda folder: (folder / "runtime.json").stat().st_mtime)
report = next(latest.glob("*.md"))
display(Markdown(report.read_text(encoding="utf-8")))
for artifact in sorted(latest.iterdir()):
    if artifact.suffix not in {".md", ".json"}:
        continue
    encoded = base64.b64encode(artifact.read_bytes()).decode("ascii")
    name = html.escape(artifact.name, quote=True)
    display(HTML(f'<a download="{name}" href="data:application/octet-stream;base64,{encoded}">Tải {name}</a>'))
print("Artifacts above are the unchanged bytes from the completed Kaggle CPU run.")
